<a href="https://colab.research.google.com/github/hamidatbi/himalaya-glof-risk/blob/main/Hymalias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



 **PHASE 1**

In [1]:
!pip install geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 65.5 MB/s  0:00:00


In [2]:
import ee
import geemap
print("all imported")

all imported


In [3]:
ee.Authenticate()
ee.Initialize(project='himalayan-glacial-risk')
print("done")

done


In [4]:
lake=ee.Geometry.Point([88.198, 27.913])
print('lake saved')

lake saved


In [5]:
s2=ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
print("Saved")

Saved


In [6]:
before= (s2
         .filterBounds(lake)
         .filterDate('2022-10-10', '2022-11-30')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)))

after= (s2
         .filterBounds(lake)
         .filterDate('2023-10-10', '2023-11-30')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)))

print('Before images:', before.size().getInfo())
print('After images:', after.size().getInfo())

Before images: 16
After images: 14


In [7]:
before_img = before.median()
after_img = after.median()
print('done')

done


In [8]:
vis= {'bands': ['B4', 'B3', 'B2'],
      'min': 0,
      'max': 6000}

map=geemap.Map(center=[27.913, 88.198], zoom=14, basemap='Esri.WorldImagery')

map.addLayer(before_img, vis, 'In 2022')
map.addLayer(after_img, vis, 'In 2023')

map

Map(center=[27.913, 88.198], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [9]:
admin_bounds=ee.FeatureCollection("FAO/GAUL/2025/level1")
study_area = admin_bounds.filter(
    ee.Filter.Or(
        ee.Filter.inList('GAUL0_NAME', ['Bhutan', 'Nepal']),
        ee.Filter.eq('GAUL1_NAME', 'Sikkim')
    )
)

study_area.size().getInfo()

28

to see the columns:  geemap.ee_to_df(admin_bounds.limit(5))

In [ ]:
#geemap.ee_to_df(admin_bounds.limit(5))

import pandas as pd

india = admin_bounds.filter(ee.Filter.eq('GAUL0_NAME', 'India'))
df = geemap.ee_to_df(india)

#df.head()
df[df['GAUL1_NAME']=='Sikkim']

,CONTINENT,DISP_EN,FID,GAUL0_CODE,GAUL0_NAME,GAUL1_CODE,GAUL1_NAME,ISO3_CODE,MAP_CODE
24,Asia,"Sikkim, India",1325,240,India,2608,Sikkim,IND,IND


In [10]:
map=geemap.Map(center=[28, 86.5], zoom=6, basemap='Esri.WorldImagery')
map.addLayer(study_area, {}, 'Study area')

aoi=study_area.union()
map.addLayer(aoi, {'color': 'red'}, 'AOI')

map

Map(center=[28, 86.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [11]:
copernicus=ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1")
dem=copernicus.select('DEM')
mosaic=dem.mosaic()
dem_aoi=mosaic.clip(aoi.geometry())


vis= {'palette':['green', 'yellow', 'brown', 'white'],
      'min': 0,
      'max': 8000}

map=geemap.Map(center=[27.913, 88.198], zoom=7, basemap='Esri.WorldImagery')
map.addLayer(dem_aoi, vis, 'DEM of study area')

map

Map(center=[27.913, 88.198], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

In [12]:
high_elev=dem_aoi.gt(4000)  #elevations more than 4000m: 1yes and 0no
masked_hel=dem_aoi.updateMask(high_elev)

map.addLayer(masked_hel,vis,'Elevations above 4k meters')
map

Map(bottom=14036.0, center=[27.913, 88.198], controls=(WidgetControl(options=['position', 'transparent_bg'], p…

**PHASE 2**

In [24]:
s2_2025= (s2
         .filterBounds(aoi.geometry())
         .filterDate('2025-09-15', '2025-11-30')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)))

#print('Lakes 2025 images count:', s2_2025.size().getInfo())

final_s2_2025=s2_2025.median().clip(aoi.geometry())

vis= {'bands': ['B4', 'B3', 'B2'],
      'min': 0,
      'max': 3000}

map=geemap.Map(center=[28, 86.5], zoom=7, basemap='Esri.WorldImagery')
map.addLayer(final_s2_2025, vis, "Median satellite image of Sep-Nov 2025")

map


Map(center=[28, 86.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

In [25]:
s2_2025_high_elev=final_s2_2025.updateMask(high_elev)
vis_high_elev={'bands': ['B4', 'B3', 'B2'],
               'min': 0,
               'max': 10000}

map.addLayer(s2_2025_high_elev, vis_high_elev, "High elevation image of Sep-Nov 2025")
map


Map(bottom=14027.0, center=[28, 86.5], controls=(WidgetControl(options=['position', 'transparent_bg'], positio…

In [27]:
s2_2025_high_elev=final_s2_2025.updateMask(high_elev)
ndwi=s2_2025_high_elev.normalizedDifference(['B3', 'B8'])

ndwi_vis={'palette': ['white', 'blue'],
          'min': -1,
          'max': 1}

map.addLayer(ndwi, ndwi_vis, "NDWI 2025")
map

Map(bottom=27802.0, center=[27.771051193172273, 87.61047363281251], controls=(WidgetControl(options=['position…

In [29]:
ndsi=s2_2025_high_elev.normalizedDifference(['B3', 'B11'])

slope_n= ee.Terrain.slope(dem_aoi)
slope_high_elev=slope_n.updateMask(high_elev)
dem_high_elev=dem_aoi.updateMask(high_elev)

ndwi=ndwi.rename('NDWI')
ndsi=ndsi.rename('NDSI')

features=s2_2025_high_elev.addBands(ndwi).addBands(ndsi).addBands(slope_high_elev).addBands(dem_high_elev)

features.bandNames().getInfo()


['B1',
 'B2',
 'B3',
 'B4',
 'B5',
 'B6',
 'B7',
 'B8',
 'B8A',
 'B9',
 'B11',
 'B12',
 'AOT',
 'WVP',
 'SCL',
 'TCI_R',
 'TCI_G',
 'TCI_B',
 'MSK_CLDPRB',
 'MSK_SNWPRB',
 'QA10',
 'QA20',
 'QA60',
 'MSK_CLASSI_OPAQUE',
 'MSK_CLASSI_CIRRUS',
 'MSK_CLASSI_SNOW_ICE',
 'NDWI',
 'NDSI',
 'slope',
 'DEM']

In [30]:
final_features=features.select(['B2','B3','B4','B8','B11','B12','NDWI','NDSI','slope','DEM'])
final_features.bandNames().getInfo()


['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDWI', 'NDSI', 'slope', 'DEM']